In [1]:
import pandas as pd
import numpy as np
import warnings
import random
import re
import os
import json
warnings.filterwarnings("ignore")

from tqdm import tqdm
from copy import deepcopy

PREFIX = "../../data/1022"

In [2]:
# system_msg = "You are a behavioral neurologist. Follow these conditions when generating the answer:\n1. Your task is to answer the question by basing your response solely on the 'Patient' section. 'Diagnosis' section is only provided for your reference. It is not patient history. Use it only to verify your analysis.\n2. You are allowed to use the 'Diagnosis' section of the input indirectly to guide your answers. But, do not reference any details from the 'Diagnosis' section in your responses.\n3. Do not use terms like 'clinician', 'clinician's assessment', 'clinical diagnosis', 'deemed by a clinician' or any other phrases indicating a confirmed diagnosis. Instead use phrases such as 'I diagnose', 'I assess', and so on indicating you are the one making the assessment.\n4. Answer responsibly, avoiding overconfidence, and encourage the user to consult a healthcare professional for advice.\n5. Your responses should not include any recommendations.\n6. When any information is missing, assume it was not collected.\n7. Please verify your answers with the 'Diagnosis' section in the input."
# print(system_msg)

## Start cleaning generated responses

In [2]:
nacc = pd.read_csv(f"{PREFIX}/csv_to_txt/all_nacc_csv_to_txt.csv")

In [4]:
len(nacc['NACCID'].unique())

50962

In [5]:
print(len(nacc[nacc['NACCUDSD'] != 2]['NACCID'].unique()))

49938


In [2]:
import json

data_1 = []
with open(f"{PREFIX}/diagnosis/nacc_unique_with_llama_summaries_1.json", "r") as file:
    for line in file:
        try:
            line = json.loads(line)
            if line['NACCUDSD'] != 2:
                data_1.append(line)
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON: {e}")
            
data_2 = []
with open(f"{PREFIX}/diagnosis/nacc_unique_with_llama_summaries_2.json", "r") as file:
    for line in file:
        
        try:
            line = json.loads(line)
            if line['NACCUDSD'] != 2:
                data_2.append(line)
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON: {e}")
            
data_3 = []
with open(f"{PREFIX}/diagnosis/nacc_unique_with_llama_summaries_3.json", "r") as file:
    for line in file:
        try:
            line = json.loads(line)
            if line['NACCUDSD'] != 2:
                data_3.append(line)
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON: {e}")
            
data_4 = []
with open(f"{PREFIX}/diagnosis/nacc_unique_with_llama_summaries_4.json", "r") as file:
    for line in file:
        try:
            line = json.loads(line)
            if line['NACCUDSD'] != 2:
                data_4.append(line)
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON: {e}")

data = data_1 + data_2 + data_3 + data_4
print(f'Length of data Before cleaning: {len(data)}')

In [7]:
print(f'Length of data before cleaning: {len(data_1)}')
print(f'Length of data before cleaning: {len(data_2)}')
print(f'Length of data before cleaning: {len(data_3)}')
print(f'Length of data before cleaning: {len(data_4)}')

Length of data before cleaning: 158678
Length of data before cleaning: 157120
Length of data before cleaning: 157581
Length of data before cleaning: 122624


In [5]:
print(data_1[0]['NACCUDSD'])
print(data_1[0]['answer'])

1
Based on the provided patient information, I assess that this 44-year-old female subject appears to be functioning within normal limits cognitively. She has a high level of independence and is able to live independently with her spouse. Her education level is 18 years, and she has a normal body mass index (BMI) of 35.8. 

The subject's neuropsychological battery summary scores indicate that her cognitive status is deemed normal for her age. Her performance on various tests such as the Craft Story Recall, Trail Making Test, and Montreal Cognitive Assessment (MoCA) suggests that she has no significant cognitive impairments.

The subject's Geriatric Depression Scale (GDS) score is 1 out of 15, indicating minimal depressive symptoms. The Functional Assessment Scale (FAS) also shows that she has no difficulty with daily activities such as paying attention to TV programs, playing games, keeping track of current events, or traveling.

The subject's physical examination reveals no abnormal n

In [8]:
ids = set()
ids_list = list()
question_set = set()

for val in data:
    ids.add(val['NACCID'])
    ids_list.append(val['NACCID'])
    question_set.add(val['instruction'])

In [9]:
question_set

{"Assess the patient's Cognitive status at UDS visit using the provided information, categorizing it as normal cognition, mild cognitive impairment, or dementia.",
 "Determine the differential diagnosis for dementia that best aligns with the patient's symptoms and presentation. List the three most probable diagnoses without specifying 'top 3' in your answer.",
 "Evaluate whether psychiatric disorders are contributing to this patient's neurological condition and explain your reasoning.",
 'Given the following patient information, compile a brief and efficient hypothesis-driven neurological history in a paragraph.',
 'Using the provided information, determine the Primary etiologic diagnosis for this patient.'}

In [10]:
print(len(ids))

49938


#### Identify answers with mentions of clinician's assessment or diagnosis section

In [11]:
clinician_diagnosis_section_answers = {}
j = 0
for i, entry in enumerate(data):
    if ("clinician" in entry['answer']) or ("clinical diagnosis" in entry['answer']) or ("diagnosis section" in entry['answer']) or ("diagnoses section" in entry['answer']):
        clinician_diagnosis_section_answers[i] = entry

In [12]:
len(clinician_diagnosis_section_answers)

2976

#### Identify answers with wrong diagnosis

In [13]:
wrong_diagnosis = {}
j = 0
for i, entry in enumerate(data):
    if (entry['NACCUDSD'] == 1):
        # if 'dementia' in entry['answer'] or 'mild cognitive impairment' in entry['answer']:
        if "is experiencing some cognitive decline" in entry['answer'] or 'is experiencing some cognitive difficulties' in entry['answer'].lower() or "experiencing mild cognitive impairment" in entry['answer'] or "experiencing a mild cognitive impairment" in entry['answer'] or "I assess the patient's cognitive status at the visit as mild cognitive impairment" in entry['answer'] or "I diagnose this patient with mild cognitive impairment" in entry['answer'] or "is experiencing cognitive decline" in entry['answer'] or 'possibly a mild cognitive impairment' in entry['answer'] or "I assess the patient's cognitive status at the visit as unable to be determined due to incomplete neuropsychological examination results" in entry['answer'] or "I assess the patient's cognitive status at the visit as unable to determine due to incomplete neuropsychological examination results" in entry['answer'] or "some level of cognitive decline" in entry['answer'] or "experiencing mild cognitive symptoms" in entry['answer']:
                    # continue
            if "appears to be functioning within normal" in entry['answer'] or "is deemed normal for his age" in entry['answer'] or "cognitive status remains normal" in entry['answer'] or "cognitively intact" in entry['answer'] or "does not meet the criteria for dementia or MCI" in entry['answer'] or "but does not meet the criteria for mild cognitive impairment or dementia" in entry['answer'] or "does not meet the criteria for mild cognitive impairment (MCI) or dementia" in entry['answer']  or "does not exhibit any signs of dementia or mild cognitive impairment" in entry['answer'] or "does not meet the criteria for dementia or mild cognitive impairment" in entry['answer'] or "does not meet criteria for dementia or MCI" in entry['answer'] or "this patient may be experiencing mild cognitive impairment (MCI), possibly related to depression or age-related cognitive decline" in entry['answer']  or "I assess the patient's cognitive status at the visit as normal cognition" in entry['answer'] or "no significant cognition impairment" in entry['answer'] or "high level of cognitive function" in entry['answer'] or "living independently" in entry['answer'] or "high level of independence" in entry['answer'] or "normal cognitive status" in entry['answer'] or "functioning within normal limits" in entry['answer'] or "functioning independently" in entry['answer'] or "live independently" in entry['answer'] or "lives independently" in entry['answer'] or "deemed to be normal" in entry['answer'] or "deemed to be normal" in entry['answer'] or "no evidence of mild cognitive impairment (MCI) or dementia" in entry['answer'] or "normal cognition" in entry['answer'] or "I do not identify any significant neurological or psychiatric concerns" in entry['answer'] or "no evidence of cognitive impairment or dementia" in entry['answer'] or "no significant cognitive impairment" in entry['answer']:
                continue
            wrong_diagnosis[i] = entry
        
    if (entry['NACCUDSD'] == 3):
        if "I assess the patient's cognitive status at the visit as normal cognition" in entry['answer'] or "which is consistent with his normal cognition" in entry['answer'].lower() or "which is consistent with normal cognition" in entry['answer'].lower() or "I assess the patient's cognitive status as normal cognition" in entry['answer'] or "I assess the patient's cognitive status at the visit as dementia" in entry['answer'] or "I assess the patient's cognitive status as dementia" in entry['answer'] or "I assess the patient's cognitive status at the visit as unable to be determined due to incomplete neuropsychological examination results" in entry['answer'] or "I assess the patient's cognitive status at the visit as unable to determine due to incomplete neuropsychological examination results" in entry['answer']:
            
            wrong_diagnosis[i] = entry
        
    if (entry['NACCUDSD'] == 4):
        if "I assess the patient's cognitive status at the visit as normal cognition" in entry['answer'] or "which is consistent with his normal cognition" in entry['answer'].lower() or "which is consistent with normal cognition" in entry['answer'].lower() or "I assess the patient's cognitive status as normal cognition" in entry['answer'] or "I assess the patient's cognitive status at the visit as mild cognitive impairment" in entry['answer'] or "I assess the patient's cognitive status as mild cognitive impairment" in entry['answer'] or "I assess the patient's cognitive status at the visit as unable to be determined due to incomplete neuropsychological examination results" in entry['answer']:
            # print(entry['CDRGLOB'], entry['NACCUDSD'], entry['NACCETPR'])
            # # print("Question:", entry['instruction'])
            # print()
            # print("Answer:", entry['answer'])
            # # print(entry['patient_summary'])
            # # print(entry['diag_summary'])
            # print()
            # print()
            # break
            wrong_diagnosis[i] = entry

In [14]:
len(wrong_diagnosis) 

7238

#### Clean responses

In [15]:
phrases = {
    "as per your instruction": "",
    " (as per your instruction)": "",
    " (D1 form)": "",
    " provided for reference only": "",
    ", their cognitive performance suggests some level of impairment that is not severe enough to be classified as normal cognition": "",
}

for i in range(len(data)):
    for x, y in phrases.items():
        data[i]['answer'] = data[i]['answer'].replace(x,y)

In [16]:
# phrases = {
#     "as per your instruction": "",
#     " (as per your instruction)": "",
#     " (D1 form)": "",
#     "was deemed normal cognition by the clinician's assessment": "seems normal",
#     "was deemed normal cognition by the clinician": "",
#     "Although not explicitly mentioned in the provided information as a possible diagnosis by other clinicians (as per your instruction): I have included it here due": "Due",
#     "and without further information from other sections that could be used for reference only for verification purposes in this case": '',
#     " alone without referencing any information from the 'Diagnosis' section for verification purposes only;": '',
#     "Although not explicitly mentioned in the diagnosis section provided for reference only for verification purposes but based": "Based",
#     "or clarification from the 'Diagnosis' section for verification purposes only ": '',
#     " for reference only for verification purposes but not in my response)": '',
#     "Considering these findings and based on my assessment as a behavioral neurologist without referencing any information from the 'Diagnosis' section directly but using it indirectly for verification purposes only.": "",
#     "Although there is no mention of Alzheimer's disease in the presumptive etiologic diagnoses section provided for reference only for verification purposes. However based": "Based",
#     " from the 'Diagnosis' section for verification purposes only": ",",
#     "Although not explicitly mentioned in the diagnosis section provided for reference only purposes but based": '',
#     " in the diagnosis section": '',
#     "in the presumptive etiologic diagnosis section": '',
#     " according to the diagnosis section": "",
#     " according to their diagnosis section (MCI)": "",
#     "However, it is essential to note that depression is listed as a non-contributing cause of cognitive impairment in the provided diagnosis section.": "",
#     " from the diagnosis section": "",
#     "However, the diagnosis section does not indicate Alzheimer's disease as the primary cause of cognitive impairment.": "",
#     'listed as "Depression" in the provided diagnosis section': "depression",
#     "However, it is essential to note that CFS is not explicitly listed as a contributing cause of cognitive impairment in the provided diagnosis section.": "",
#     " mentioned in the other presumptive etiologic diagnosis section": "",
#     ", as indicated by their MCI type being non-amnestic MCI-single domain with visuospatial being affected and their presumptive etiologic diagnosis being Lewy body disease in the provided diagnosis section": "with non-amnestic MCI-single domain with visuospatial being affected",
#     "The patient's cognitive status is deemed normal for their age based on the neuropsychological examination, but they do have mild cognitive impairment (MCI)": "The patient has mild cognitive impairment (MCI)",
#     "However, the patient's anxiety disorder diagnosis is noted in the primary etiologic diagnosis section.": "",
#     " or any other specific etiology mentioned in her diagnosis section": "",
#     " primary etiologic diagnosis of FTLD not otherwise specified (NOS) as indicated by the presumptive etiologic diagnosis section suggests that the": "",
#     ", which is consistent with his primary etiologic diagnosis as per the presumptive etiologic diagnosis section": "",
#     " according to the presumptive etiologic diagnosis section": "",
#     " reported in their presumptive etiologic diagnosis section": "",
#     " as per the presumptive etiologic diagnosis section": "",
#     "according to the presumptive etiologic diagnosis section of the input data provided for reference purposes only. However": ",",
#     " presumptive etiologic diagnosis section": "",
#     ", which was mentioned as another presumptive etiologic diagnosis in the other presumptive etiologic diagnosis section": "",
#     " or TBI diagnosis in their presumptive etiologic diagnosis section but they have been reported to have TBI in their presumptive etiologic diagnosis section which": ",TBI",
#     " in the provided diagnosis section": "",
#     "is listed as": "is",
#     "Although the diagnosis section indicates that NPH was assessed and found not present, I consider it": "I consider NPH",
#     "However, the diagnosis section indicates that the primary contributing cause of cognitive impairment is not possible vascular dementia.": "",
#     "differential diagnosis section provided for reference only": "provided information",
#     " provided for reference only": "",
#     "His presence of behavioral frontotemporal dementia (bvFTD) is present as per the diagnosis section.": "",
#     "However, given the absence of other significant psychiatric symptoms and the presence of cognitive impairment due to vascular brain injury or vascular dementia including stroke as per the diagnosis section for reference only.": "",
#     "Considering these findings and based on my assessment alone without any reference to the diagnosis section provided for my reference only.": "",
#     " as indicated by their presumptive etiologic diagnosis of anxiety in their diagnosis section": "",
#     "It is also worth noting that the patient has been diagnosed with probable vascular dementia as per the provided diagnosis section.": "",
#     "and the presence of dementia syndrome - Posterior cortical atrophy syndrome on the diagnosis section": "",
#     "differential diagnosis section of the provided information": "provided information",
#     ", given the presence of dementia syndrome - Lewy body dementia syndrome in their diagnosis section": "",
#     "Although the diagnosis section indicates that NPH was assessed and found not present, I will consider it": "I will consider NPH",
#     ", as they have a primary etiologic diagnosis of Alzheimer's disease listed in their diagnosis section": "",
#     ", which is confirmed by the diagnosis section": "",
#     " and MCI domain affected - language in his diagnosis section": "",
#     "The presence of Lewy body disease (LBD) in the primary etiologic diagnosis section suggests that this": "Lewy body disease (LBD)",
#     "Although the diagnosis section indicates that NPH was assessed and found not present, I include it": "I include NPH",
#     "However, considering their age and cognitive impairment status as dementia syndrome - Amnestic multidomain dementia syndrome is present in their diagnosis section which indicates there might be an underlying psychiatric component contributing to their condition": "",
#     ", which is confirmed by the diagnosis section": "",
#     " (as per the diagnosis section)": "",
#     ", as well as a positive presence of behavioral frontotemporal dementia (bvFTD) in the diagnosis section": "",
#     ", their cognitive performance suggests some level of impairment that is not severe enough to be classified as normal cognition": "",
    
#     # "diagnosis section indicates": "diagnosis section suggests",
#     # "provided diagnosis section": "provided information",
#     # "diagnosis section": "provided information",
#     # "differential diagnosis section": "provided information",
    
    
# }
# for i in range(len(data)):
#     for x, y in phrases.items():
#         data[i]['answer'] = data[i]['answer'].replace(x,y)
    

## Identify the indices of the responses with repeated sentences

In [17]:
import nltk
from collections import defaultdict

# Ensure you have the Punkt tokenizer models downloaded
nltk.download('punkt') #, download_dir='/projectnb/vkolagrp/skowshik/.cache/punkt')


[nltk_data] Downloading package punkt to
[nltk_data]     /usr3/graduate/skowshik/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [18]:
def find_highly_repeated_sentences(responses):
    nums = ['1.', '2.', '3.', '4.', '5.', '6.', '7.', '8.', '9.', '*']
    highly_repeated = defaultdict(list)
    
    for i, response in enumerate(responses):
        # Tokenize the response into sentences
        sentences = nltk.sent_tokenize(response)
        sentence_counts = defaultdict(int)
        
        # Count each sentence
        for sentence in sentences:
            if sentence in nums:
                continue
            sentence_counts[sentence] += 1
        
        # Check for sentences with more than two repetitions
        for sentence, count in sentence_counts.items():
            if count > 4:
                highly_repeated[i].append((sentence, count))

    return highly_repeated

def modify_response_to_keep_single_repeated(responses):
    modified_responses = {}
    highly_repeated_indices = defaultdict(list)
    
    for i, response in enumerate(responses):
        # Tokenize the response into sentences
        sentences = nltk.sent_tokenize(response)
        sentence_counts = defaultdict(int)
        
        # Count each sentence
        for sentence in sentences:
            sentence_counts[sentence] += 1
            
        # Check for sentences with more than two repetitions
        for sentence, count in sentence_counts.items():
            if count > 4:
                highly_repeated_indices[i].append((sentence, count))

        # Find sentences repeated more than four times
        highly_repeated = {sentence for sentence, count in sentence_counts.items() if count > 4}

        # Reconstruct the response keeping only one instance of each highly repeated sentence
        modified_response = []
        seen = set()
        for sentence in sentences:
            if sentence in highly_repeated:
                if sentence not in seen:
                    modified_response.append(sentence)
                    seen.add(sentence)
            else:
                modified_response.append(sentence)

        # Join modified response back into a single string
        if i in highly_repeated_indices:
            modified_responses[i] = ' '.join(modified_response)
        
    return modified_responses, highly_repeated_indices

def get_repeated_indices(key, data_list):
    # new_data = deepcopy(data_list)
    
    responses = []
    for i in range(len(data_list)):
        responses.append(data_list[i][key])

    repeat_indices = []
    # Find sentences repeated more than 4 times
    highly_repeated_indices = find_highly_repeated_sentences(responses)
    # modified_responses_dict, highly_repeated_indices = modify_response_to_keep_single_repeated(responses)
    
    # Print the results
    for index, repeat in highly_repeated_indices.items():
        repeat_indices.append(index)
    
    return highly_repeated_indices, repeat_indices
    
    # # Print the results
    # for index, modified_response in modified_responses_dict.items():
    #     new_data[index][key] = modified_response
    #     # print(new_data[i][key])
    #     # print(index, "modified")

    # return new_data, repeat_indices


In [19]:
# patient_modified_data, patient_repeat_indices = get_repeated_indices('patient_LLAMA_SUMMARY', data)
# diag_modified_data, diag_repeat_indices = get_repeated_indices('diag_LLAMA_SUMMARY', data)
repeats, diag_repeat_indices = get_repeated_indices('answer', data)

In [20]:
print(f"Number of patient responses repeated {len(diag_repeat_indices)}")
# print(f"Number of diagnosis responses repeated {len(diag_repeat_indices)}")

Number of patient responses repeated 0


In [24]:
repeats

defaultdict(list, {})

In [25]:
# ind = 4
# print(data[diag_repeat_indices[ind]]['instruction'], data[diag_repeat_indices[ind]]['NACCID'])
# print(data[diag_repeat_indices[ind]]['answer'])

## Clean the responses

In [26]:
# Delete the responses by ordering them in descending order
def delete_by_indices(lst, indices):
    # Sort the indices in descending order to avoid index shift issues
    for index in sorted(indices, reverse=True):
        if index < len(lst):
            del lst[index]

In [27]:
indices_to_delete = []
for ind in diag_repeat_indices:
    indices_to_delete.append(ind)

In [28]:
delete_by_indices(data, indices_to_delete)
delete_by_indices(data, wrong_diagnosis)
delete_by_indices(data, clinician_diagnosis_section_answers)

In [29]:
print(f'Length of data After cleaning: {len(data)}')

Length of data After cleaning: 585849


#### Add MRI paths to the patient summary

In [154]:
# def get_mri_or_emb_dict(val):
#     mris_found = defaultdict(int)
#     if val == 'mri':
#         data_path = '/projectnb/vkolagrp/datasets/NACC/MRI/skullstripped'
#         save_path = 'nacc_mri_paths.json'
#         extension = '.nii'
#     else:
#         data_path = '/projectnb/vkolagrp/datasets/NACC/MRI/swin_emb'
#         save_path = 'nacc_emb_paths.json'
#         extension = '.npy'
#     j = 6
#     def extract_id(name):
#         # This regex matches any characters following "fname-" up until the first occurrence of "_mo"
#         match = re.search(r'fname-(.*?)_mo', name)
#         if match:
#             return match.group(1)    # Returns the captured group which is the part of the string you need
#         else:
#             match = re.search(r'fname-(.*?)_nodate', name)
#             if match:
#                 return match.group(1) 
#         return None

#     mri_dict = {}
#     # for dirpath, dirs, files in tqdm(os.walk('/projectnb/vkolagrp/datasets/NACC/MRI/skullstripped')): 
#     for dirpath, dirs, files in tqdm(os.walk(data_path)): 
#         if len(files) == 0:
#             continue
#         for filename in files:
#             fname = os.path.join(dirpath, filename)
#             path_list = fname.split('/')
#             naccid = path_list[j+1]
#             # print(path_list)
#             zip_name = extract_id(path_list[j+2])
#             modality = path_list[j+3]
#             acquisition = path_list[j+4]
#             mri_name = path_list[j+5]
#             if mri_name in mris_found:
#                 continue
#             mris_found[mri_name] += 1
#             # print(naccid, zip_name, modality, acquisition, mri_name)
            
#             if modality == 'Unknown' or (not mri_name.endswith(extension)):
#                 continue
#             # if naccid not in mri_dict:
#             #     mri_dict[naccid] = {}
            
#             if zip_name + 'ni' not in mri_dict:
#                 mri_dict[zip_name + 'ni'] = {}
            
#             mri_dict[zip_name + 'ni'][modality] = {"acquisition": acquisition, "mri_name": fname}
            
                
                

#     # with open(save_path, "w") as outfile:
#     #     json.dump(mri_dict, outfile, indent=4, sort_keys=False)
        
#     return mri_dict, mris_found

In [155]:
# emb_dict, mris_found = get_mri_or_emb_dict(val='emb')

In [156]:
# modified_data = deepcopy(data)
# for i, entry in enumerate(modified_data):
#     zip_name = nacc.loc[nacc['NACCID'] == entry['NACCID']]['NACCMRFI'].iloc[0]
#     if isinstance(zip_name, str):
#         mod_zip_name = zip_name.replace(".zip", "ni")
#         if mod_zip_name in emb_dict:
#             mri_text = ''
#             for seq, info in emb_dict[mod_zip_name].items():
#                 if "diffusion" in seq.lower():
#                     continue
#                 mri_text += f"{seq} MRI image: {info['mri_name']} "
            
#             print(mri_text)
#             modified_data[i]['patient_LLAMA_SUMMARY'] += f"\n\n**MRI imaging**:\n\n{mri_text}"
#             # print(modified_data[i]['patient_LLAMA_SUMMARY'])
#             # break

## Save train and test datasets

In [30]:
nacc_np = pd.read_csv(f'/projectnb/vkolagrp/datasets/NACC/csv/processed/investigator_ftldlbd_merged_neuropath_vnum_unique_nacc65.csv')
ct_path = f"{PREFIX}/ct/ct_data.json"       
with open(ct_path, 'r', encoding='utf-8') as json_file:
    data_ct = json.load(json_file)
    
eeg_path = f"{PREFIX}/eeg/eeg_data.json"       
with open(eeg_path, 'r', encoding='utf-8') as json_file:
    data_eeg = json.load(json_file)

In [31]:
len(nacc_np)

34240

In [32]:
new_data = {}
i = 1
for entry in data:
    new_data[f"NACC_{i}"] = entry
    i += 1

In [33]:
data_train = {}
data_test = {}
train = 1
test = 1
np_ids = set(list(nacc_np['NACCID']))
for entry in data:
    if (entry['NACCID'] in np_ids):
        data_test[f"NACC_{test}"] = entry
        test += 1
    else:
        data_train[f"NACC_{train}"] = entry
        train += 1
        
data_train.update(data_ct)
data_train.update(data_eeg)

In [34]:
print(f"Train split: {len(data_train)}")
print(f"Test split: {len(data_test)}")

Train split: 453551
Test split: 134493


In [ ]:
json_name_train = f"{PREFIX}/train_data/train.json"
with open(json_name_train, 'w', encoding='utf-8') as json_file:
    json.dump(data_train, json_file, indent=4, sort_keys=False)

json_name_test = f"{PREFIX}/train_data/test.json"
with open(json_name_test, 'w', encoding='utf-8') as json_file:
    json.dump(data_test, json_file, indent=4, sort_keys=False)